# nn_ft_transformer_hpo — FT-Transformer + Optuna HPO (코랩 GPU)

**목적**: 부스팅 family 외부 베이스 (NN) 추가하여 stacking 잔차 다양성 확보. plateau corr 0.99+ 한계를 깰 가능성.

**구성**:
- 모델: **FT-Transformer** (Numerical Feature Tokenizer + CLS token + Multi-head Attention)
- 학습 단위: **unit-level** (die→unit aggregate 6 funcs: mean/std/range/min/max/median → ~3,408 features)
- 학습 패턴: **단일 회귀** (broadcast 아님, unit y 직접). target = `log1p(y_unit)`, inference `expm1+clip≥0`
- Loss: **MSE / Tweedie (1.2/1.5/1.7)** — Optuna categorical
- 전처리: 03b log1p preset PP (missing 0.5, indicator 0.25, post 0.99)
- Optuna 탐색: 모델 HP (4) + 학습 HP (3) + loss (1) + (옵션) y=0 weight (1) = **9 axis**
- KFold: 5 unit-level shuffle SEED=42

**격리**: `4_output/_temp/nn_ft/` 신규.

**비교 기준 (단일 base val RMSE)**:
- 03b (path A die): val=0.005718, test=0.008417
- 03f (path A unit-agg): val=0.005742
- path B (reverse): val=0.005709
- zit_only: val=0.005709
- 본 NN 베이스가 plateau 안 (0.0057x) 들어오면 stacking 후보. 잔차 corr < 0.95 면 강한 가치.

## 1. 환경 + import (Colab GPU / Local 공통)

In [ ]:
import os, sys, json, math, random, pickle

# === 결정성 (torch import 전에 환경변수 설정) ===
DETERMINISTIC = True   # True면 cudnn deterministic + 알고리즘 고정. 약 5~15% 속도 손실
if DETERMINISTIC:
    # use_deterministic_algorithms(True) + cuBLAS GEMM이 결정적이려면 필요
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/preprocess.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

import optuna
from sklearn.model_selection import KFold

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.aggregate import aggregate_to_unit

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
from final.modules import preprocess
from final.modules.scaling import HybridScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision('high')   # TF32 matmul (정밀도 손실 미미)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if DETERMINISTIC:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as _e:
            print(f'[deterministic] use_deterministic_algorithms 설정 실패: {_e}')
    else:
        torch.backends.cudnn.benchmark = True

USE_AMP            = (DEVICE == 'cuda')          # bf16 autocast
USE_COMPILE_HPO    = False                       # HPO 단계는 compile OFF (매 fold 재컴파일 방지)
USE_COMPILE_REFIT  = (DEVICE == 'cuda') and (not DETERMINISTIC)  # compile은 비결정성 유발 가능

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DEVICE = {DEVICE}')
print(f'PyTorch = {torch.__version__}, CUDA available = {torch.cuda.is_available()}')
if DEVICE == 'cuda':
    print(f'GPU = {torch.cuda.get_device_name(0)}')
    print(f'AMP(bf16)={USE_AMP}, compile(HPO/refit)={USE_COMPILE_HPO}/{USE_COMPILE_REFIT}, '
          f'TF32=ON, deterministic={DETERMINISTIC}')


## 2. 설정

In [ ]:
EXP_ID   = 'nn-ft-transformer-002-hybrid'
EXP_MEMO = 'FT-Transformer + HybridScaler + 기존 DB best HP refit (HPO skip) + 결정성 모드'
USER     = 'jh'

# === HPO 토글 ===
# True  : 30 trial Optuna HPO + 5-fold refit (원본 흐름)
# False : HPO 건너뛰고 BEST_HP_FROM_DB로 5-fold refit만 (빠름, 시나리오 비교용)
RUN_HPO = False

# === 기존 best_params.json (final/nn_ft) 결과를 그대로 사용 ===
BEST_HP_FROM_DB = {
    'd_model':      64,
    'n_layers':     4,
    'dropout':      0.2829012260843204,
    'lr':           0.0024939396315541013,
    'weight_decay': 1.0345436422454988e-05,
    'batch_size':   2048,
    'loss_type':    'mse',
    'n_heads':      4,    # fixed
    'ffn_factor':   2,    # fixed
}

N_TRIALS       = 30
N_FOLDS_HPO    = 3      # HPO 단계 fold 수 (속도 우선)
N_FOLDS_REFIT  = 5      # best HP refit 단계 fold 수 (정확도 우선)
MAX_EPOCHS     = 40
PATIENCE       = 7
CLIP_Y_EXTREME = True
SAVE_FOLD_MODELS = True   # fold별 state_dict .pt 저장 (재현성)

# OUT_DIR을 분리 — StandardScaler 베이스라인(_temp/nn_ft)과 비교 가능하게
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'nn_ft_hybrid')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS (03b log1p preset 동일) ──
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}
AGG_FUNCS = ['mean', 'std', 'range', 'min', 'max', 'median']

print(f'EXP_ID={EXP_ID}')
print(f'RUN_HPO={RUN_HPO}  (False면 BEST_HP_FROM_DB로 refit만)')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS_HPO={N_FOLDS_HPO} | N_FOLDS_REFIT={N_FOLDS_REFIT}')
print(f'MAX_EPOCHS={MAX_EPOCHS} | PATIENCE={PATIENCE} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'SAVE_FOLD_MODELS={SAVE_FOLD_MODELS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'AGG_FUNCS={AGG_FUNCS}')


## 3. 데이터 로드 + Y clip + 전처리 + unit-agg + scaling

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# ── die-level cleaning ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')

# ── unit aggregate (6 funcs) ──
X_tr_df = aggregate_to_unit(xs_train_die, feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)
X_vl_df = aggregate_to_unit(xs_val_die,   feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)
X_te_df = aggregate_to_unit(xs_test_die,  feat_cols=feat_cols_clean, agg_funcs=AGG_FUNCS)

# NaN check (range/std 가 NaN 될 수 있음)
X_tr_df = X_tr_df.fillna(0.0)
X_vl_df = X_vl_df.fillna(0.0)
X_te_df = X_te_df.fillna(0.0)

# y 인덱스 순서로 정렬
X_tr_df = X_tr_df.loc[y_train_unit.index]
X_vl_df = X_vl_df.loc[y_val_unit.index]
X_te_df = X_te_df.loc[y_test_unit.index]

# ── HybridScaler (train fit, all transform) ──
# binary passthrough(nunique≤2) + |skew|>10 → Quantile + 나머지 → Power(Yeo-Johnson)
SCALER_KW = dict(
    skew_threshold=10.0,
    n_quantiles=1000,
    quantile_output='normal',
    random_state=SEED,
    binary_passthrough=True,
)
scaler = HybridScaler(**SCALER_KW).fit(X_tr_df)
X_tr_scaled_df = scaler.transform(X_tr_df, inplace=False)
X_vl_scaled_df = scaler.transform(X_vl_df, inplace=False)
X_te_scaled_df = scaler.transform(X_te_df, inplace=False)

X_train_s = X_tr_scaled_df.values.astype(np.float32)
X_val_s   = X_vl_scaled_df.values.astype(np.float32)
X_test_s  = X_te_scaled_df.values.astype(np.float32)

y_train = y_train_unit.values.astype(np.float32)
y_val   = y_val_unit.values.astype(np.float32)
y_test  = y_test_unit.values.astype(np.float32)

N_FEATURES = X_train_s.shape[1]
SCALER_STATS = {
    'type':              'HybridScaler',
    'skew_threshold':    SCALER_KW['skew_threshold'],
    'n_quantiles':       SCALER_KW['n_quantiles'],
    'quantile_output':   SCALER_KW['quantile_output'],
    'binary_passthrough': SCALER_KW['binary_passthrough'],
    'binary_n':          len(scaler.binary_cols_),
    'quantile_n':        len(scaler.quantile_cols_),
    'power_n':           len(scaler.power_cols_),
}
print(f'\n[unit-agg + HybridScale] N_FEATURES={N_FEATURES}')
print(f'  X_train_s: {X_train_s.shape}, val: {X_val_s.shape}, test: {X_test_s.shape}')
print(f'  scaler 분배: binary={SCALER_STATS["binary_n"]}, '
      f'quantile={SCALER_STATS["quantile_n"]}, power={SCALER_STATS["power_n"]}')
print(f'  y_train pos ratio={(y_train > 0).mean():.4f}')


## 4. FT-Transformer 모델 + Loss 정의

**FT-Transformer** (Yandex 2021):
1. Numerical Feature Tokenizer: 각 feature 를 d_model 차원 token 으로 변환 (`x[i] · W[i] + b[i]`)
2. CLS token prepend
3. N stacked Transformer blocks (MHA + FFN)
4. CLS embedding → MLP head → 1-d output

**Loss**:
- MSE: `F.mse_loss(pred_log, log1p(y))`
- Tweedie: `expm1(pred_log).clamp(eps) → tweedie_deviance(·, y, power)`

In [ ]:
class NumericalTokenizer(nn.Module):
    """각 numerical feature → d_model dim token. (B, F) → (B, F, d_model)."""
    def __init__(self, n_features, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_model))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
    def forward(self, x):
        # x: (B, F)
        return x.unsqueeze(-1) * self.weight + self.bias


class FTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, ffn_factor):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_factor),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_factor, d_model),
        )
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        # PreNorm style + Flash Attention (need_weights=False → SDPA 활성)
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(a)
        h = self.norm2(x)
        x = x + self.dropout(self.ffn(h))
        return x


class FTTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=3,
                 dropout=0.1, ffn_factor=2):
        super().__init__()
        self.tokenizer = NumericalTokenizer(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)
        self.blocks = nn.ModuleList([
            FTBlock(d_model, n_heads, dropout, ffn_factor)
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
    def forward(self, x):
        # x: (B, F)
        h = self.tokenizer(x)                            # (B, F, D)
        cls = self.cls_token.expand(h.size(0), -1, -1)   # (B, 1, D)
        h = torch.cat([cls, h], dim=1)                    # (B, F+1, D)
        for blk in self.blocks:
            h = blk(h)
        return self.head(h[:, 0]).squeeze(-1)             # (B,)


def loss_mse(pred_log, y_orig, **kwargs):
    target_log = torch.log1p(y_orig)
    return F.mse_loss(pred_log, target_log)


def loss_tweedie(pred_log, y_orig, power=1.5, **kwargs):
    """Tweedie deviance for 1 < power < 2 (compound Poisson-Gamma).
    pred_log → expm1+clamp 으로 양수 mu 만들어서 deviance 계산.
    """
    mu = torch.clamp(torch.expm1(pred_log), min=1e-6)
    a = y_orig * (mu ** (1 - power)) / (1 - power)
    b = (mu ** (2 - power)) / (2 - power)
    return torch.mean(b - a)


def get_loss_fn(loss_type):
    if loss_type == 'mse':
        return loss_mse
    elif loss_type.startswith('tweedie'):
        power = float(loss_type.split('_')[1])
        return lambda pred_log, y_orig: loss_tweedie(pred_log, y_orig, power=power)
    else:
        raise ValueError(f'unknown loss: {loss_type}')


def predict_y(pred_log_np):
    """log1p prediction → original y. clip>=0."""
    return np.clip(np.expm1(pred_log_np), 0.0, None)


print('FT-Transformer + Loss 정의 완료 (Flash Attention via need_weights=False)')

## 5. 1-fold 학습 함수 (early stopping + val RMSE)

In [ ]:
def _rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(true)) ** 2)))


def _amp_ctx():
    """bf16 autocast (수치 결과 거의 동일, A100/H100 네이티브)."""
    if USE_AMP:
        return torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()


def _predict_batched(model, X_t, bs=4096):
    """긴 시퀀스에서 OOM 방지하기 위한 배치 추론. forward bf16 → 반환 fp32."""
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, X_t.size(0), bs):
            with _amp_ctx():
                pred = model(X_t[i:i+bs])
            outs.append(pred.float().cpu().numpy())
    return np.concatenate(outs, axis=0)


def _train_one_fold_impl(X_tr, y_tr, X_vl, y_vl, hp, X_others=None,
                          max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=False,
                          use_compile=False):
    """1-fold 학습 + 예측 (bf16 forward + fp32 loss + manual batch + optional compile)."""
    model = FTTransformer(
        n_features=N_FEATURES,
        d_model=hp['d_model'],
        n_heads=hp['n_heads'],
        n_layers=hp['n_layers'],
        dropout=hp['dropout'],
        ffn_factor=hp['ffn_factor'],
    ).to(DEVICE)

    if use_compile:
        try:
            model = torch.compile(model, mode='default', dynamic=True)
        except Exception as _e:
            if verbose:
                print(f'    compile skipped: {_e}')

    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
    loss_fn = get_loss_fn(hp['loss_type'])

    Xtr_t = torch.from_numpy(X_tr).to(DEVICE)
    ytr_t = torch.from_numpy(y_tr).to(DEVICE)
    Xvl_t = torch.from_numpy(X_vl).to(DEVICE)
    others_t = [torch.from_numpy(X).to(DEVICE) for X in (X_others or [])]

    n_train = X_tr.shape[0]
    bs = hp['batch_size']

    best_val_rmse = float('inf')
    best_oof = None
    best_others = None
    bad_count = 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        # 수동 batch iteration (DataLoader 오버헤드 제거, 데이터 이미 GPU)
        perm = torch.randperm(n_train, device=DEVICE)
        epoch_loss = 0.0
        for i in range(0, n_train, bs):
            idx = perm[i:i+bs]
            xb = Xtr_t[idx]
            yb = ytr_t[idx]
            opt.zero_grad(set_to_none=True)
            # forward 만 bf16, loss 는 fp32 (작은 health 값 정밀도 보존)
            with _amp_ctx():
                pred_log = model(xb)
            loss = loss_fn(pred_log.float(), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= n_train

        # eval (배치 추론 — 큰 val 셋에서도 안전)
        vl_pred_log = _predict_batched(model, Xvl_t, bs=bs * 4)
        vl_pred = predict_y(vl_pred_log)
        vl_rmse = _rmse(vl_pred, y_vl)

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f'    ep{epoch:3d}  loss={epoch_loss:.6f}  val={vl_rmse:.6f}')

        if vl_rmse < best_val_rmse - 1e-7:
            best_val_rmse = vl_rmse
            best_oof = vl_pred
            best_epoch = epoch
            best_others = [predict_y(_predict_batched(model, X_o, bs=bs * 4)) for X_o in others_t]
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= patience:
                break

    if best_oof is None:
        # fallback: 마지막 epoch 결과
        best_oof = predict_y(_predict_batched(model, Xvl_t, bs=bs * 4))
        best_others = [predict_y(_predict_batched(model, X_o, bs=bs * 4)) for X_o in others_t]
        best_val_rmse = _rmse(best_oof, y_vl)

    del model, opt, Xtr_t, ytr_t, Xvl_t, others_t
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return best_oof, best_others, best_val_rmse, best_epoch


def train_one_fold(X_tr, y_tr, X_vl, y_vl, hp, X_others=None,
                    max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=False,
                    min_bs=32, use_compile=False):
    """OOM fallback 래퍼: CUDA OOM 발생 시 batch_size 절반으로 줄여 재시도.
    use_compile: HPO에서는 False, refit에서는 True 권장.
    """
    hp = dict(hp)
    orig_bs = hp['batch_size']
    while hp['batch_size'] >= min_bs:
        try:
            return _train_one_fold_impl(
                X_tr, y_tr, X_vl, y_vl, hp,
                X_others=X_others,
                max_epochs=max_epochs, patience=patience, verbose=verbose,
                use_compile=use_compile,
            )
        except torch.cuda.OutOfMemoryError as _e:
            torch.cuda.empty_cache()
            new_bs = hp['batch_size'] // 2
            print(f'  [OOM] bs {hp["batch_size"]} → {new_bs} fallback '
                  f'(orig={orig_bs}, hp d_model={hp["d_model"]}/n_layers={hp["n_layers"]}/ffn={hp["ffn_factor"]})')
            hp['batch_size'] = new_bs
    raise RuntimeError(f'OOM 지속 (bs<{min_bs}). hp={hp}')


print('train_one_fold 정의 완료 (bf16 + fp32 loss + manual batch + optional compile + OOM fallback)')

## 6. Optuna HPO

각 trial 당 **3-fold OOF RMSE** 최소화. **MedianPruner** 가 fold 단위로 가지치기 → 나쁜 trial 30~50% 시간 절약.

**탐색축 7개**: d_model, n_layers, dropout, lr, weight_decay, batch_size, loss_type.
**고정축 2개**: n_heads=4, ffn_factor=2 (덜 중요, trial 효율 위해 고정).

**compile**: HPO 단계는 **OFF** (매 fold 재컴파일 방지, 30~45분 절약), refit 단계만 ON.

In [ ]:
import time

# Refit 단계: 5-fold (RUN_HPO 여부와 무관하게 항상 필요)
kf_refit = KFold(n_splits=N_FOLDS_REFIT, shuffle=True, random_state=SEED)
FOLDS_REFIT = list(kf_refit.split(np.arange(len(y_train))))

study = None  # RUN_HPO=False 분기에서 cell-save가 참조

if RUN_HPO:
    # HPO 단계: 3-fold (속도 우선)
    kf_hpo = KFold(n_splits=N_FOLDS_HPO, shuffle=True, random_state=SEED)
    FOLDS_HPO = list(kf_hpo.split(np.arange(len(y_train))))


    def objective(trial):
        # === 탐색축 7개 (n_heads=4, ffn_factor=2 고정) ===
        hp = {
            'd_model':      trial.suggest_categorical('d_model', [32, 48, 64, 96]),
            'n_heads':      4,
            'n_layers':     trial.suggest_int('n_layers', 2, 4),
            'dropout':      trial.suggest_float('dropout', 0.05, 0.4),
            'ffn_factor':   2,
            'lr':           trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size':   trial.suggest_categorical('batch_size', [256, 512, 1024, 2048]),
            'loss_type':    trial.suggest_categorical(
                'loss_type', ['mse', 'tweedie_1.2', 'tweedie_1.5', 'tweedie_1.7']
            ),
        }

        oof_pred = np.zeros(len(y_train), dtype=np.float32)
        fold_val_rmses = []
        t0 = time.time()
        for fi, (tr_idx, vl_idx) in enumerate(FOLDS_HPO):
            oof_fold, _, vl_rmse, n_ep = train_one_fold(
                X_train_s[tr_idx], y_train[tr_idx],
                X_train_s[vl_idx], y_train[vl_idx],
                hp,
                X_others=None,
                use_compile=USE_COMPILE_HPO,
            )
            oof_pred[vl_idx] = oof_fold
            fold_val_rmses.append(vl_rmse)
            trial.report(vl_rmse, fi)
            if trial.should_prune():
                trial.set_user_attr('pruned_at_fold', fi)
                trial.set_user_attr('elapsed_s', time.time() - t0)
                raise optuna.TrialPruned()

        oof_rmse_score = _rmse(oof_pred, y_train)
        trial.set_user_attr('oof_rmse', oof_rmse_score)
        trial.set_user_attr('mean_fold_val', float(np.mean(fold_val_rmses)))
        trial.set_user_attr('elapsed_s', time.time() - t0)
        return oof_rmse_score


    study = optuna.create_study(
        direction='minimize',
        study_name=EXP_ID,
        storage=f'sqlite:///{DB_PATH}',
        load_if_exists=False,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    )
    study.set_user_attr('exp_memo', EXP_MEMO)
    study.set_user_attr('device', DEVICE)
    study.set_user_attr('n_folds_hpo', N_FOLDS_HPO)
    study.set_user_attr('n_folds_refit', N_FOLDS_REFIT)
    study.set_user_attr('compile_hpo', USE_COMPILE_HPO)
    study.set_user_attr('compile_refit', USE_COMPILE_REFIT)
    study.set_user_attr('scaler', 'HybridScaler')
    study.set_user_attr('deterministic', DETERMINISTIC)

    print(f'=== Optuna HPO 시작 (N_TRIALS={N_TRIALS}, N_FOLDS_HPO={N_FOLDS_HPO}, '
          f'MAX_EPOCHS={MAX_EPOCHS}, PATIENCE={PATIENCE}) ===')
    print(f'    탐색축: 7개 (d_model, n_layers, dropout, lr, weight_decay, batch_size, loss_type)')
    print(f'    고정축: n_heads=4, ffn_factor=2')
    print(f'    pruner: MedianPruner (startup=5, warmup_steps=0)')
    print(f'    compile: HPO={USE_COMPILE_HPO}, refit={USE_COMPILE_REFIT}')
    t_total = time.time()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f'\n[HPO 완료] {time.time()-t_total:.0f}s')
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    print(f'  trials: complete={n_complete}, pruned={n_pruned}')
    print(f'  best OOF RMSE = {study.best_value:.6f}')
    print(f'  best params:')
    for k, v in study.best_trial.params.items():
        print(f'    {k:15s} = {v}')
else:
    print('=== RUN_HPO=False → HPO 건너뜀. BEST_HP_FROM_DB로 바로 refit ===')
    print(f'  BEST_HP_FROM_DB:')
    for k, v in BEST_HP_FROM_DB.items():
        print(f'    {k:15s} = {v}')


## 7. Best params 5-fold refit + OOF/val/test

In [ ]:
# best HP 결정: HPO 결과 우선, 없으면 DB 하드코딩 사용
if RUN_HPO and study is not None:
    best_hp = dict(study.best_trial.params)
    best_hp['n_heads']    = 4
    best_hp['ffn_factor'] = 2
    HP_SOURCE = 'optuna_current_run'
else:
    best_hp = dict(BEST_HP_FROM_DB)
    HP_SOURCE = 'fixed_from_previous_db (final/nn_ft/best_params.json)'

print(f'best_hp source: {HP_SOURCE}')

oof_pred_arr  = np.zeros(len(y_train), dtype=np.float32)
val_pred_arr  = np.zeros(len(y_val),   dtype=np.float32)
test_pred_arr = np.zeros(len(y_test),  dtype=np.float32)
fold_records = []
fold_state_paths = []

print(f'=== Refit (best params) {N_FOLDS_REFIT}-fold '
      f'(compile={USE_COMPILE_REFIT}, deterministic={DETERMINISTIC}, '
      f'save_states={SAVE_FOLD_MODELS}) ===')
t0 = time.time()

if SAVE_FOLD_MODELS:
    # ── 직접 학습 루프 (학습 1회로 oof + val/test 예측 + state 저장 모두 처리) ──
    fold_state_dir = os.path.join(OUT_DIR, 'fold_states')
    os.makedirs(fold_state_dir, exist_ok=True)

    for fi, (tr_idx, vl_idx) in enumerate(FOLDS_REFIT):
        # fold별 시드 고정 (재현성)
        torch.manual_seed(SEED + fi)
        if DEVICE == 'cuda':
            torch.cuda.manual_seed_all(SEED + fi)

        model = FTTransformer(
            n_features=N_FEATURES,
            d_model=best_hp['d_model'],
            n_heads=best_hp['n_heads'],
            n_layers=best_hp['n_layers'],
            dropout=best_hp['dropout'],
            ffn_factor=best_hp['ffn_factor'],
        ).to(DEVICE)
        opt = torch.optim.AdamW(
            model.parameters(),
            lr=best_hp['lr'],
            weight_decay=best_hp['weight_decay'],
        )
        loss_fn = get_loss_fn(best_hp['loss_type'])

        Xtr_t   = torch.from_numpy(X_train_s[tr_idx]).to(DEVICE)
        ytr_t   = torch.from_numpy(y_train[tr_idx]).to(DEVICE)
        Xvl_t   = torch.from_numpy(X_train_s[vl_idx]).to(DEVICE)
        Xval_t  = torch.from_numpy(X_val_s).to(DEVICE)
        Xtest_t = torch.from_numpy(X_test_s).to(DEVICE)
        yvl     = y_train[vl_idx]

        n_train_fold = Xtr_t.shape[0]
        bs_fold = best_hp['batch_size']
        best_vl = float('inf')
        best_state = None
        best_oof_pred = None
        best_val_pred = None
        best_test_pred = None
        best_epoch = 0
        bad = 0

        for ep in range(1, MAX_EPOCHS + 1):
            model.train()
            perm = torch.randperm(n_train_fold, device=DEVICE)
            epoch_loss = 0.0
            for i in range(0, n_train_fold, bs_fold):
                idx = perm[i:i+bs_fold]
                opt.zero_grad(set_to_none=True)
                with _amp_ctx():
                    pred_log = model(Xtr_t[idx])
                loss = loss_fn(pred_log.float(), ytr_t[idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                epoch_loss += loss.item() * idx.size(0)
            epoch_loss /= n_train_fold

            vl_pred_log = _predict_batched(model, Xvl_t, bs=bs_fold * 4)
            vl_pred = predict_y(vl_pred_log)
            vl_rmse_ep = _rmse(vl_pred, yvl)

            if (fi == 0) and (ep == 1 or ep % 5 == 0):
                print(f'    ep{ep:3d}  loss={epoch_loss:.6f}  val={vl_rmse_ep:.6f}')

            if vl_rmse_ep < best_vl - 1e-7:
                best_vl = vl_rmse_ep
                best_oof_pred = vl_pred
                best_val_pred = predict_y(_predict_batched(model, Xval_t,  bs=bs_fold * 4))
                best_test_pred = predict_y(_predict_batched(model, Xtest_t, bs=bs_fold * 4))
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_epoch = ep
                bad = 0
            else:
                bad += 1
                if bad >= PATIENCE:
                    break

        if best_oof_pred is None:
            # fallback: 학습이 한 번도 개선 안 된 경우 마지막 epoch 결과 사용
            best_oof_pred = predict_y(_predict_batched(model, Xvl_t,  bs=bs_fold * 4))
            best_val_pred = predict_y(_predict_batched(model, Xval_t, bs=bs_fold * 4))
            best_test_pred = predict_y(_predict_batched(model, Xtest_t, bs=bs_fold * 4))
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_vl = _rmse(best_oof_pred, yvl)

        # 누적
        oof_pred_arr[vl_idx] = best_oof_pred
        val_pred_arr  += best_val_pred  / N_FOLDS_REFIT
        test_pred_arr += best_test_pred / N_FOLDS_REFIT
        fold_records.append({'fold': fi+1, 'val_rmse': best_vl, 'epochs': best_epoch})

        # state_dict 저장
        state_path = os.path.join(fold_state_dir, f'fold_{fi+1}.pt')
        torch.save({
            'state_dict':    best_state,
            'best_val_rmse': best_vl,
            'best_hp':       best_hp,
            'n_features':    int(N_FEATURES),
            'fold_idx':      fi + 1,
            'seed_used':     SEED + fi,
            'best_epoch':    best_epoch,
        }, state_path)
        fold_state_paths.append(state_path)

        del model, opt, Xtr_t, ytr_t, Xvl_t, Xval_t, Xtest_t
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        print(f'  fold {fi+1}/{N_FOLDS_REFIT}: vl_rmse={best_vl:.6f}  '
              f'n_epochs={best_epoch}  state→{os.path.basename(state_path)}  '
              f'elapsed={time.time()-t0:.0f}s')
else:
    # ── 기존 train_one_fold 흐름 (state 저장 안 함, OOM fallback 활성) ──
    for fi, (tr_idx, vl_idx) in enumerate(FOLDS_REFIT):
        oof_fold, others, vl_rmse, n_ep = train_one_fold(
            X_train_s[tr_idx], y_train[tr_idx],
            X_train_s[vl_idx], y_train[vl_idx],
            best_hp,
            X_others=[X_val_s, X_test_s],
            verbose=(fi == 0),
            use_compile=USE_COMPILE_REFIT,
        )
        oof_pred_arr[vl_idx] = oof_fold
        val_pred_arr  += others[0] / N_FOLDS_REFIT
        test_pred_arr += others[1] / N_FOLDS_REFIT
        fold_records.append({'fold': fi+1, 'val_rmse': vl_rmse, 'epochs': n_ep})
        print(f'  fold {fi+1}/{N_FOLDS_REFIT}: vl_rmse={vl_rmse:.6f}  '
              f'n_epochs={n_ep}  elapsed={time.time()-t0:.0f}s')

print(f'\n[Refit 완료] {time.time()-t0:.0f}s')


## 8. RMSE 평가

In [ ]:
oof_rmse  = _rmse(oof_pred_arr,  y_train)
val_rmse  = _rmse(val_pred_arr,  y_val)
test_rmse = _rmse(test_pred_arr, y_test)

print('=' * 75)
if RUN_HPO:
    print(f'  FT-Transformer + Optuna HPO + HybridScaler — best result')
else:
    print(f'  FT-Transformer + HybridScaler (HPO skipped, fixed best HP) — result')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선 (단일 base val):')
print(f'    nn_ft (StandardScaler):            val=0.005822, test=0.008504')
print(f'    03b (path A die):                  val=0.005718, test=0.008417')
print(f'    03f (path A unit-agg):             val=0.005742')
print(f'    path B (reverse):                  val=0.005709, test=0.008412')
print(f'    zit_only:                          val=0.005709, test=0.008414')
print(f'    Stacking 11-base (val best):       val=0.005701, test=0.008408')
print('=' * 75)
if val_rmse < 0.0058:
    print(f'  → plateau 영역 안. stacking pool 추가 후보 (잔차 corr 검증 필요).')
elif val_rmse < 0.006:
    print(f'  → plateau 근처. residual 패턴이 다르면 stacking 도움 가능.')
else:
    print(f'  → plateau 밖. NN 가 부스팅 family 못 따라감.')


## 9. 산출물 저장 (`_temp/nn_ft_hybrid/`)

In [ ]:
def _build_unit_df(uid, pred, y_true):
    return pd.DataFrame({
        KEY_COL:    uid,
        'pred':     pred,
        TARGET_COL: y_true,
    })

_build_unit_df(y_train_unit.index.values, oof_pred_arr,  y_train).to_csv(
    os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(y_val_unit.index.values,   val_pred_arr,  y_val).to_csv(
    os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(y_test_unit.index.values,  test_pred_arr, y_test).to_csv(
    os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# ── best_params.json (HPO 안 돌렸어도 hp_source 명시) ──
if RUN_HPO and study is not None:
    best_value = study.best_value
    best_params_dict = study.best_trial.params
else:
    best_value = None  # OOF RMSE는 meta.json에 기록
    best_params_dict = {k: v for k, v in BEST_HP_FROM_DB.items()
                        if k not in ('n_heads', 'ffn_factor')}

with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'hp_source':    HP_SOURCE,
        'best_value':   best_value,
        'best_params':  best_params_dict,
        'fixed_params': {'n_heads': 4, 'ffn_factor': 2},
        'fold_records': fold_records,
    }, f, indent=2, ensure_ascii=False, default=str)

# ── HybridScaler pickle (예측 파이프라인 재사용) ──
with open(os.path.join(OUT_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# ── 환경 / 버전 정보 ──
ENV_INFO = {
    'torch_version':       torch.__version__,
    'cuda_version':        getattr(torch.version, 'cuda', None),
    'cudnn_version':       (torch.backends.cudnn.version()
                            if (DEVICE == 'cuda' and torch.backends.cudnn.is_available())
                            else None),
    'gpu_name':            (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None),
    'cudnn_benchmark':     bool(torch.backends.cudnn.benchmark),
    'cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
    'tf32_matmul':         bool(torch.backends.cuda.matmul.allow_tf32),
}

if RUN_HPO and study is not None:
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
else:
    n_pruned   = 0
    n_complete = 0

meta = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'model':             'FT-Transformer (numerical, unit-agg 6 funcs)',
    'training_level':    'unit (post-aggregate)',
    'target_transform':  'log1p',
    'aggregation':       AGG_FUNCS,
    'n_features':        int(N_FEATURES),
    'run_hpo':           RUN_HPO,
    'hp_source':         HP_SOURCE,
    'n_trials':          N_TRIALS if RUN_HPO else 0,
    'n_complete_trials': n_complete,
    'n_pruned_trials':   n_pruned,
    'n_folds_hpo':       N_FOLDS_HPO,
    'n_folds_refit':     N_FOLDS_REFIT,
    'max_epochs':        MAX_EPOCHS,
    'patience':          PATIENCE,
    'device':            DEVICE,
    'use_amp':           USE_AMP,
    'use_compile_hpo':   USE_COMPILE_HPO,
    'use_compile_refit': USE_COMPILE_REFIT,
    'deterministic':     DETERMINISTIC,
    'pruner':            'MedianPruner(n_startup_trials=5, n_warmup_steps=0)',
    'fixed_params':      {'n_heads': 4, 'ffn_factor': 2},
    'optimizer':         'AdamW',
    'scheduler':         None,
    'grad_clip_norm':    1.0,
    'loss_type':         best_hp['loss_type'],
    'scaler':            SCALER_STATS,
    'env':               ENV_INFO,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'best_params':       best_params_dict,
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'SEED':              int(SEED),
    'save_fold_models':  SAVE_FOLD_MODELS,
    'fold_state_paths':  fold_state_paths,
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f_)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {f_:35s}  {sz:>10,.1f} KB')
    else:
        # fold_states/ 같은 하위 디렉토리
        sub_files = os.listdir(p)
        sub_total = sum(os.path.getsize(os.path.join(p, x)) for x in sub_files) / 1024
        print(f'  {f_+"/":35s}  {sub_total:>10,.1f} KB ({len(sub_files)} files)')

# ── Colab → 로컬 자동 다운로드 ──
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'nn_ft_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass


## 10. 요약

In [ ]:
print('=' * 75)
print(f' FT-Transformer + HybridScaler — 결과 요약')
print('=' * 75)
print(f'  EXP_ID                : {EXP_ID}')
print(f'  RUN_HPO / hp_source   : {RUN_HPO} / {HP_SOURCE}')
print(f'  device / AMP          : {DEVICE} / {USE_AMP}')
print(f'  compile (HPO/refit)   : {USE_COMPILE_HPO} / {USE_COMPILE_REFIT}')
print(f'  deterministic         : {DETERMINISTIC}')
print(f'  scaler                : HybridScaler (binary={SCALER_STATS["binary_n"]}, '
      f'quantile={SCALER_STATS["quantile_n"]}, power={SCALER_STATS["power_n"]})')
print(f'  N_FEATURES            : {N_FEATURES} (= {len(feat_cols_clean)} × {len(AGG_FUNCS)})')

if study is not None:
    print(f'  N_TRIALS / pruned     : {N_TRIALS} / {n_pruned} (complete={n_complete})')
    print(f'  N_FOLDS_HPO/REFIT     : {N_FOLDS_HPO} / {N_FOLDS_REFIT}')
    print(f'  fixed: n_heads=4, ffn_factor=2')
    print(f'  best HP (Optuna):')
    for k, v in study.best_trial.params.items():
        print(f'    {k:15s} = {v}')
else:
    print(f'  N_FOLDS_REFIT         : {N_FOLDS_REFIT} (HPO skipped)')
    print(f'  fixed: n_heads=4, ffn_factor=2')
    print(f'  best HP (from DB):')
    for k, v in BEST_HP_FROM_DB.items():
        print(f'    {k:15s} = {v}')

print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → oof_unit.csv 외부 stacking 노트북에서 base 로 사용 가능')
print(f'  → 잔차 corr 측정으로 plateau 풀의 다양성 추가 여부 검증 권장')
if SAVE_FOLD_MODELS:
    print(f'  → fold_states/fold_{{1..{N_FOLDS_REFIT}}}.pt 저장됨 (재현/추론 가능)')
print('=' * 75)
